In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai__Key = '' # @param {type:"string", placeholder:"Required: paste your Civitai API key"}
HF_Read_Token = '' # @param {type:"string", placeholder:"Optional: paste your Hugging Face read token"}
Mount_GDrive = 'No' # @param ["No", "Yes"]
Drive_Root = '/content/drive/MyDrive/Segsmaker' # @param {type:"string"}

from pathlib import Path
import json
import os
import shlex
import time

SEGSMaker_DRIVE_ENABLED = Mount_GDrive == 'Yes'
SEGSMaker_DRIVE_ROOT = Path(Drive_Root).expanduser()

if SEGSMaker_DRIVE_ENABLED:
    from google.colab import drive
    drive.mount('/content/drive')
    SEGSMaker_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

def segsmaker_is_path(value):
    return value is not None and str(value).strip() not in {'', 'None'}

def segsmaker_drive_root():
    if not SEGSMaker_DRIVE_ENABLED:
        return None
    SEGSMaker_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    return SEGSMaker_DRIVE_ROOT

def segsmaker_drive_kind_path(kind):
    root = segsmaker_drive_root()
    if root is None:
        return None
    path = root / kind
    path.mkdir(parents=True, exist_ok=True)
    return path

def segsmaker_asset_path(kind, fallback_path, load_from_drive=False):
    if load_from_drive and SEGSMaker_DRIVE_ENABLED:
        drive_path = segsmaker_drive_kind_path(kind)
        if drive_path is not None:
            return drive_path

    if not segsmaker_is_path(fallback_path):
        return None

    path = Path(fallback_path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def segsmaker_safe_symlink(link_path, target_path):
    if not SEGSMaker_DRIVE_ENABLED:
        return
    if not segsmaker_is_path(link_path) or not segsmaker_is_path(target_path):
        return

    link = Path(link_path)
    target = Path(target_path)
    target.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)

    if link.exists() or link.is_symlink():
        try:
            if link.is_symlink() and link.resolve() == target.resolve():
                return
        except Exception:
            pass
        print(f'Skipping existing path: {link}')
        return

    link.symlink_to(target, target_is_directory=True)
    print(f'Linked Drive folder: {link} -> {target}')

def segsmaker_clean_value(value):
    return str(value or '').strip()

def segsmaker_make_download_line(url_value, target_path):
    value = segsmaker_clean_value(url_value)
    if not value:
        return None
    if not segsmaker_is_path(target_path):
        return None
    return f'{value} {Path(target_path)}'

def segsmaker_magic_worker(kind, line, cwd, label, queue):
    try:
        if cwd:
            os.chdir(str(cwd))
        if kind == 'download':
            from nenen88 import download
            download(line)
        elif kind == 'clone':
            from nenen88 import clone
            clone(line)
        else:
            raise ValueError(f'Unknown job kind: {kind}')
        queue.put((label, True, ''))
    except Exception as exc:
        queue.put((label, False, repr(exc)))

def segsmaker_run_jobs(jobs, parallel=False, max_workers=3):
    jobs = [job for job in jobs if job.get('line')]
    if not jobs:
        print('No valid input to process.')
        return []

    workers = max(1, int(max_workers or 1))
    workers = min(workers, len(jobs))

    if not parallel or workers == 1 or len(jobs) == 1:
        results = []
        for job in jobs:
            label = job.get('label', job['kind'])
            try:
                segsmaker_magic_worker(
                    job['kind'],
                    job['line'],
                    job.get('cwd'),
                    label,
                    type('InlineQueue', (), {'put': lambda self, value: results.append(value)})()
                )
            except Exception as exc:
                results.append((label, False, repr(exc)))
        return results

    # ThreadPoolExecutor intentionally not used: %download changes process cwd.
    # Forked processes isolate cwd and make parallel downloads safer in Colab.
    import multiprocessing as mp

    try:
        ctx = mp.get_context('fork')
    except ValueError:
        ctx = mp.get_context()

    queue = ctx.Queue()
    pending = list(jobs)
    running = []
    results = []

    while pending or running:
        while pending and len(running) < workers:
            job = pending.pop(0)
            process = ctx.Process(
                target=segsmaker_magic_worker,
                args=(
                    job['kind'],
                    job['line'],
                    job.get('cwd'),
                    job.get('label', job['kind']),
                    queue,
                ),
            )
            process.start()
            running.append((process, job))

        still_running = []
        for process, job in running:
            if process.is_alive():
                still_running.append((process, job))
            else:
                process.join()
        running = still_running

        while not queue.empty():
            results.append(queue.get())

        if pending or running:
            time.sleep(0.25)

    while not queue.empty():
        results.append(queue.get())

    return results

def segsmaker_report_results(results):
    if not results:
        return
    ok = [label for label, success, _ in results if success]
    failed = [(label, message) for label, success, message in results if not success]

    print(f'Completed: {len(ok)}')
    if failed:
        print(f'Failed: {len(failed)}')
        for label, message in failed:
            print(f'- {label}: {message}')

if SEGSMaker_DRIVE_ENABLED:
    drive_links = {
        'checkpoint': (CKPT, 'drive-checkpoint'),
        'lora': (LORA, 'drive-lora'),
        'vae': (VAE, 'drive-vae'),
        'embeddings': (Embeddings, 'drive-embeddings'),
        'upscalers': (Upscalers, 'drive-upscalers'),
        'unet': (UNET, 'drive-unet'),
        'clip': (CLIP, 'drive-clip'),
        'text_encoders': (TE, 'drive-text-encoders'),
    }

    for kind, (base_path, link_name) in drive_links.items():
        if segsmaker_is_path(base_path):
            segsmaker_safe_symlink(Path(base_path) / link_name, segsmaker_drive_kind_path(kind))


In [ ]:
# @title <b><font color='orange'>Model Downloader - 5 Checkpoint + 5 LoRA + VAE</font></b> {"display-mode":"form"}

Checkpoint_1 = '' # @param {type:"string", placeholder:"Checkpoint URL or URL custom_name.safetensors"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Lora_1 = '' # @param {type:"string", placeholder:"LoRA URL or URL custom_name.safetensors"}
Lora_2 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_3 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_4 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_5 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
VAE_URL = '' # @param {type:"string", placeholder:"VAE URL or leave empty"}
Load_From_Drive = False # @param {type:"boolean"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

jobs = []
skipped = []

checkpoint_target = segsmaker_asset_path('checkpoint', CKPT, Load_From_Drive)
lora_target = segsmaker_asset_path('lora', LORA, Load_From_Drive)
vae_target = segsmaker_asset_path('vae', VAE, Load_From_Drive)

for label, value in [
    ('Checkpoint_1', Checkpoint_1),
    ('Checkpoint_2', Checkpoint_2),
    ('Checkpoint_3', Checkpoint_3),
    ('Checkpoint_4', Checkpoint_4),
    ('Checkpoint_5', Checkpoint_5),
]:
    line = segsmaker_make_download_line(value, checkpoint_target)
    if line:
        jobs.append({'kind': 'download', 'line': line, 'label': label})
    else:
        skipped.append(label)

for label, value in [
    ('Lora_1', Lora_1),
    ('Lora_2', Lora_2),
    ('Lora_3', Lora_3),
    ('Lora_4', Lora_4),
    ('Lora_5', Lora_5),
]:
    line = segsmaker_make_download_line(value, lora_target)
    if line:
        jobs.append({'kind': 'download', 'line': line, 'label': label})
    else:
        skipped.append(label)

vae_line = segsmaker_make_download_line(VAE_URL, vae_target)
if vae_line:
    jobs.append({'kind': 'download', 'line': vae_line, 'label': 'VAE_URL'})
else:
    skipped.append('VAE_URL')

print(f'Skipped empty or unavailable fields: {len(skipped)}')
results = segsmaker_run_jobs(jobs, parallel=Parallel_Download, max_workers=Max_Workers)
segsmaker_report_results(results)


In [ ]:
# @title <b><font color='orange'>Extra Assets - Extensions, Embeddings, Upscalers</font></b> {"display-mode":"form"}

Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Embedding_1 = '' # @param {type:"string", placeholder:"Embedding URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"Embedding URL or leave empty"}
Embedding_3 = '' # @param {type:"string", placeholder:"Embedding URL or leave empty"}
Upscaler_1 = '' # @param {type:"string", placeholder:"Upscaler URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"Upscaler URL or leave empty"}
Upscaler_3 = '' # @param {type:"string", placeholder:"Upscaler URL or leave empty"}
Assets_Load_From_Drive = False # @param {type:"boolean"}
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

jobs = []
skipped = []

extension_target = Path(Extensions) if segsmaker_is_path(Extensions) else None
embedding_target = segsmaker_asset_path('embeddings', Embeddings, Assets_Load_From_Drive)
upscaler_target = segsmaker_asset_path('upscalers', Upscalers, Assets_Load_From_Drive)

for label, value in [
    ('Extension_1', Extension_1),
    ('Extension_2', Extension_2),
    ('Extension_3', Extension_3),
    ('Extension_4', Extension_4),
    ('Extension_5', Extension_5),
]:
    cleaned = segsmaker_clean_value(value)
    if cleaned and extension_target is not None:
        extension_target.mkdir(parents=True, exist_ok=True)
        jobs.append({'kind': 'clone', 'line': cleaned, 'cwd': str(extension_target), 'label': label})
    else:
        skipped.append(label)

for label, value in [
    ('Embedding_1', Embedding_1),
    ('Embedding_2', Embedding_2),
    ('Embedding_3', Embedding_3),
]:
    line = segsmaker_make_download_line(value, embedding_target)
    if line:
        jobs.append({'kind': 'download', 'line': line, 'label': label})
    else:
        skipped.append(label)

for label, value in [
    ('Upscaler_1', Upscaler_1),
    ('Upscaler_2', Upscaler_2),
    ('Upscaler_3', Upscaler_3),
]:
    line = segsmaker_make_download_line(value, upscaler_target)
    if line:
        jobs.append({'kind': 'download', 'line': line, 'label': label})
    else:
        skipped.append(label)

if extension_target is None:
    print('Extensions/custom nodes path is unavailable for this WebUI; extension fields were skipped.')
if upscaler_target is None:
    print('Upscaler path is unavailable for this WebUI; upscaler fields were skipped.')

print(f'Skipped empty or unavailable fields: {len(skipped)}')
results = segsmaker_run_jobs(jobs, parallel=Assets_Parallel_Download, max_workers=Assets_Max_Workers)
segsmaker_report_results(results)


In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}

FLUX_Variant = 'Custom URLs' # @param ["Custom URLs", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
FLUX_Unet = '' # @param {type:"string", placeholder:"FLUX unet/diffusion model URL or leave empty"}
FLUX_Clip_L = '' # @param {type:"string", placeholder:"clip_l URL or leave empty"}
FLUX_T5XXL = '' # @param {type:"string", placeholder:"t5xxl text encoder URL or leave empty"}
FLUX_VAE = '' # @param {type:"string", placeholder:"FLUX VAE URL or leave empty"}
FLUX_Load_From_Drive = False # @param {type:"boolean"}
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:4, step:1}

jobs = []
skipped = []

print(f'FLUX variant selected: {FLUX_Variant}')
print('Use the URL fields below for exact files you want. Gated Hugging Face files require HF_Read_Token in Cell 1.')

flux_unet_target = segsmaker_asset_path('unet', UNET, FLUX_Load_From_Drive)
flux_clip_target = segsmaker_asset_path('clip', CLIP, FLUX_Load_From_Drive)
flux_t5_target = segsmaker_asset_path('text_encoders', TE, FLUX_Load_From_Drive)
flux_vae_target = segsmaker_asset_path('vae', VAE, FLUX_Load_From_Drive)

for label, value, target in [
    ('FLUX_Unet', FLUX_Unet, flux_unet_target),
    ('FLUX_Clip_L', FLUX_Clip_L, flux_clip_target),
    ('FLUX_T5XXL', FLUX_T5XXL, flux_t5_target),
    ('FLUX_VAE', FLUX_VAE, flux_vae_target),
]:
    line = segsmaker_make_download_line(value, target)
    if line:
        jobs.append({'kind': 'download', 'line': line, 'label': label})
    else:
        skipped.append(label)
        if segsmaker_clean_value(value) and target is None:
            print(f'{label} skipped because the current WebUI does not expose a compatible target folder.')

print(f'Skipped empty or unavailable fields: {len(skipped)}')
results = segsmaker_run_jobs(jobs, parallel=Parallel_FLUX_Download, max_workers=FLUX_Max_Workers)
segsmaker_report_results(results)


In [ ]:
# @title <b><font color='orange'>ControlNet Downloader Widget</font></b> {"display-mode":"form"}
''' Controlnet '''
%run $Controlnet_Widget


In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}

print('Select the same WebUI that you installed in the first cell.')

Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = '' # @param {type:"string", placeholder:"Optional Ngrok token"}
Zrok_Token = '' # @param {type:"string", placeholder:"Optional Zrok token"}
Extra_Args = '' # @param {type:"string", placeholder:"Leave empty to use recommended defaults"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

from pathlib import Path
import json
import shlex

default_args = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

try:
    from KANDANG import HOMEPATH
except Exception:
    HOMEPATH = '/content'

marking_file = Path(HOMEPATH) / 'gutris1' / 'marking.json'
installed_ui = None
if marking_file.exists():
    try:
        installed_ui = json.loads(marking_file.read_text()).get('ui')
    except Exception:
        installed_ui = None

if installed_ui and installed_ui != Software:
    print(f'Warning: installed WebUI is {installed_ui}, but launcher selection is {Software}. The installed WebUI will be used by segsmaker.py.')

launch_args = segsmaker_clean_value(Extra_Args) or default_args.get(Software, '')
run_args = []

if Skip_ComfyUI_Check:
    run_args.append('--skip-comfyui-check')
if Skip_Widget:
    run_args.append('--skip-widget')
if segsmaker_clean_value(Ngrok_Token):
    run_args.extend(['--N', shlex.quote(segsmaker_clean_value(Ngrok_Token))])
if segsmaker_clean_value(Zrok_Token):
    run_args.extend(['--Z', shlex.quote(segsmaker_clean_value(Zrok_Token))])
if launch_args:
    run_args.append(launch_args)

%cd -q $WebUI
get_ipython().run_line_magic('run', 'segsmaker.py ' + ' '.join(run_args))
